# LangChain Tutorial (Step by Step) uding Grok

## Requirements
For this notebook, you need:
1. `langchain`, `langchain-groq`, and `python-dotenv`.
2. A free Groq account and an API key.
3. `GROQ_API_KEY` available from `.env` or from a secure prompt.

Optional in PowerShell:
`$env:GROQ_API_KEY="your_api_key_here"`

In [1]:
%pip install -U langchain langchain-groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import getpass
import importlib

# Load .env if python-dotenv is available
dotenv_spec = importlib.util.find_spec("dotenv")
if dotenv_spec is not None:
    dotenv_module = importlib.import_module("dotenv")
    dotenv_module.load_dotenv()

def _clean_key(value: str | None) -> str:
    if not value:
        return ""
    return value.strip().strip("\"").strip("'")

# Ask for the key only if it is still missing
if not os.getenv("GROQ_API_KEY"):
    try:
        entered_key = getpass.getpass("Enter GROQ_API_KEY: ")
    except Exception:
        entered_key = input("Enter GROQ_API_KEY (visible): ")
    os.environ["GROQ_API_KEY"] = _clean_key(entered_key)

if not _clean_key(os.getenv("GROQ_API_KEY")):
    raise EnvironmentError("GROQ_API_KEY not found. Add it to .env or enter it when prompted.")

print("GROQ_API_KEY detected")

GROQ_API_KEY detected


## 1) Initialize the model
Create a chat model instance that will be reused across the tutorial.

In [4]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

print("[✓] Model initialized: llama-3.1-8b-instant")

c:\Users\ders1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


[✓] Model initialized: llama-3.1-8b-instant


## 2) Direct invocation with messages
Send a system instruction and user message directly to the model.

This step shows the most basic way to call a chat model in LangChain.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a helpful weather assistant. Provide brief, clear responses."),
    HumanMessage(content="What's the weather like in San Francisco today?"),
]

response = model.invoke(messages)
print(response.content)

Hola! ¿Cómo estás hoy?


## 3) Add an output parser
A parser normalizes model output into a plain string for easier downstream use.

In [6]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
chain_with_parser = model | parser

parsed_text = chain_with_parser.invoke(messages)
print(parsed_text)

Hola! ¿Cómo estás hoy?


## 4) Build prompt templates
Prompt templates let you parameterize instructions (for language, input text, style, etc.).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions about {topic}."),
    ("user", "{question}"),
])

prompt_value = template.invoke({"topic": "weather", "question": "How is the temperature measured?"})
for message in prompt_value.messages:
    print(f"[{message.__class__.__name__}] {message.content}")

[SystemMessage] Translate the following into French.
[HumanMessage] Good morning!


## 5) Compose the full chain
Combine `template | model | parser` for a reusable translation pipeline.

In [ ]:
full_chain = template | model | parser

examples = [
    {"topic": "weather", "question": "What causes rain?"},
    {"topic": "geography", "question": "What is the capital of France?"},
]

for item in examples:
    out = full_chain.invoke(item)
    print(f"Q: {item['question']}")
    print(f"A: {out}\n")

German: Ich liebe Programmieren!

Here's a breakdown of the translation:

- "I love" is translated to "Ich liebe"
- "programming" is translated to "Programmieren"
Italian: Il tempo è bellissimo oggi.


## 6) Stream model output
Streaming is useful for chat-like UX because tokens appear as they are generated.

In [ ]:
print("Streaming response:\n")
for chunk in full_chain.stream({"topic": "climate", "question": "Why is the sky blue?"}):
    print(chunk, end="", flush=True)
print()

Output: Inteligência Artificial está mudando o mundo.


## Next step
From here you can adapt the same pattern to a RAG pipeline by replacing the user text with retrieved context + question.

In [10]:
print("Tutorial completed ")

Tutorial completed 


In [ ]:
# Optional quick check
full_chain.invoke({"topic": "weather", "question": "What's the difference between weather and climate?"})

'Hasta mañana.'